# 32 · 不花钱的改进：多一路排序、多一类病人、多一路病例

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → 18 疾病介绍（RAG） ✅ → 19 基因工具 ✅ → 20 大考 ✅ → 21 正式评测 ✅ → 22 做成服务 ✅ → 23 和别人比 ✅ → 24 和大模型比 ✅ → 25 和 DeepRare 比 ✅ → 26 读病历 ✅ → 27 交付 ✅ → 28 时间外验证 ✅ → 29 外部验证 ✅ → 30 离线试打分 ✅ → 31 加进 Orphanet 的病 ✅ → [32 不花钱的改进 ✅] → 33 RAG 重做 → 34 似然比打分和亚型折叠 → 35 固定流程和试跑
```

> **这一章的代码在分支 `feat/v2-design` 里。运行这本笔记不调用任何大模型**，约 15 分钟（比较排序方法约 4 分钟，比较候选路线两次、各约 4 分钟）。读 500 份病例报告那一步已经做完，下面重新运行只会确认"没有新的要读"，不花钱。

## 背景：先算清楚，再花钱

第 31 章之后，你说"每次跑总量都很费钱，先设计好"。于是做了三件事，写进了 [`docs/v2_design.md`](../v2_design.md)：

1. **三方面的文献调研**：知识、排序方法、架构和评测
2. **用已经记下的考试重新分析**（不调用模型）：v1.1 错在哪
3. **定了顺序**：能离线做的先做完（这一章），再搭新流程、在 ci-20 上试跑（下一章），最后才花钱跑大的

重新分析里最关键的一张表（详见设计文档第 1 节）：v1.1 第 1 名答错的题，正确答案在哪里。

| 来源 | 知识库里根本没有（或被遮） | 正确答案在第 2–3 名 | 不在前 10 名 |
|---|---|---|---|
| dev-200 | 35% | 26% | 62% |
| HMS | 44% | 16% | 69% |
| RAMEDIS | 0% | 27% | 28% |
| MME | 0% | 55% | 45% |

**dev-200、HMS 主要是"找不到"**。所以这一章的重点是：**让正确答案更多地出现在 agent 能看到的候选里**。

**公司里的做法**：搜索、推荐系统都分"召回"（先把可能的候选捞出来）和"精排"（再排出好坏）。召回不够，精排再好也没用；而召回的改动，大多可以离线、不花钱地量出来。

你还问了两件事：**数据够不够、工具够不够**。这一章用数据回答。

## 这一章做的 5 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 一路新的排序：单向相似度 | 自己实现，和 hpo3 逐个核对 | 旧算法惩罚症状表长的病 |
| 2 | 第五类开发病人：已发表的病例报告 | RareArena 数据集；PMC → PMID；读病历 | 开发数据太像 Phenopacket，缺"相对常见"的病 |
| 3 | 比较排序方法 | `raredx eval rankers`，事先定的规矩 | 定新流程用哪种排序、Orphanet 开不开 |
| 4 | 比较候选路线 | `raredx eval routes` | 定新流程从哪几路提名候选 |
| 5 | 结论 | — | 新流程的输入 |

**花费**：读 500 份病例报告约 $0.65（按高峰价折算；在半价时段读的，实际约一半，约 ¥2.3）。其余 0。

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
import logging, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # 一条和我们无关的提示
logging.getLogger("mlflow").setLevel(logging.WARNING)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = r.stdout + r.stderr
    print("\n".join(line for line in out.splitlines() if "INFO mlflow" not in line))
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 一路新的排序：单向相似度

### 在干什么
给"按症状排序"加一种算法：**只看病人的每个症状在这个病里找不找得到**，不再看"这个病的每个症状病人有没有"。

### 背景
第 10 章起用的算法叫 funSimAvg，它把两个方向取平均：
1. 病人的每个症状，在病里找最像的一条
2. 病的每条症状，在病人身上找最像的一个

第 31 章发现，第 2 个方向会惩罚**症状表长的病**：病人只有 9 个症状、病记了 46 条时，第 2 个方向的分很低。文献里有同样的结论：Phen2Disease 的实验里，只用第 1 个方向第 1 名 25.0%，两个方向平均 21.1%，只用第 2 个方向只有 1.6%；Exomiser 因为同一个问题改过算法。

### 用了什么技术，为什么

| 选择 | 为什么 |
|---|---|
| 自己实现（[`knowledge/oneway.py`](../../src/raredx/knowledge/oneway.py)） | hpo3 只提供两个方向都算的三种做法 |
| 预先给每个病建好"它的症状和所有上级症状"的索引 | 两个症状的 Resnik 相似度，等于它们最具体的共同上级的信息量；查一个病人症状，只要从它自己往上找，第一个出现在病的索引里的上级就是答案。一万个病，一次查询不到一秒 |
| 用测试证明算对了 | 和 hpo3 自己的"两个症状之间的相似度"逐个手算比，每个病都要相等 |

**打个比方**：招聘时只问"候选人的每项经历，岗位要求里有没有用得上的"，不再问"岗位的每项要求，候选人是不是都有"。岗位要求写得越长的公司，后一种问法越吃亏。

### 公司里什么情况怎么做
- 自己实现一个算法，**先找一个现成的、可信的实现逐个对数**，对上了再用
- 测试是"证明它对"的证据，写进代码库，每次改动自动再跑

### 执行
跑这个算法的测试（不花钱，几秒）：

In [ ]:
run("uv run pytest -q tests/knowledge/test_oneway.py")

### 结果怎么看
- 6 个测试都通过（6 个点）。其中最重要的一个：**在测试用的小知识库上，每个病的单向分数，都和用 hpo3 自己的两两相似度手算的结果相等**
- 另外三个测试确认：病有、病人没有的症状不再扣分；被遮挡的论文一样被遮；工具里可以用 `resnik/oneway` 选它

单向算法单独用好不好，第 3 步看。

### 面试怎么说
> "hpo3 only offers set similarities that average both directions, which penalise long disease profiles for patients with few findings. I implemented the one-directional Resnik score myself, from an index of each disease's ancestor closure, and tested it equal to hpo3's own term-to-term similarity on every disease before using it."

---

## 第 2 步 · 第五类开发病人：已发表的病例报告

### 在干什么
从 1 万份已发表的病例报告里，按事先定好的规矩抽 500 份，用"读病历"（第 26 章）变成 HPO 症状，作为第五类开发病人。

### 背景：数据够不够
- **训练**：现在的设计不靠训练（第 30 章证明学出来的权重换一类病人就失灵），所以不是瓶颈
- **验证**：缺的是"不像 Phenopacket"的病人。HMS 只有 88 道、MME 40 道，大约只能可靠地测出 15–20 个点以上的差别；留出集里"相对常见"的罕见病只占 5%

DeepRare 附带的病例库正好补这个缺口：RareArena 数据集里的 1 万份病例报告（出自 PubMed Central），每份有 Orphanet 的诊断、年龄、性别，**三分之一是"相对常见"的罕见病**。

两个要说清的限制：
- 这些论文早于模型的截止日，模型可能读过：**只能当开发数据，不能当考卷**
- 许可是非商用（CC BY-NC-SA 4.0）

### 用了什么技术，为什么

| 步骤 | 做法 | 为什么 |
|---|---|---|
| 登记数据 | 写进 `evals/datasets.toml`：HuggingFace 上的固定版本 `e348a29`、校验码、许可 | 和其他数据一个规矩 |
| PMC 号换 PMID | NCBI 的 ID 转换接口；对照表进 git（`evals/pmc_pmids.tsv`） | 遮挡按 PMID 进行：这份报告自己的论文，答题时不能被任何工具看到 |
| 抽样规则（写死在代码里） | 一篇论文一份；一个病最多 3 份；"相对常见"的病至少三分之一；诊断在合并后的知识库里；不要 Phenopacket Store 里有的论文；固定随机种子 | 不让某一个病、某一篇论文占太多；不和其他考题重复 |
| 读成症状 | 第 26 章的读病历：模型找出每个症状的原文，词典和检索给出 HPO 编号；只留"病人有"的症状 | 我们的系统输入是 HPO 编号 |
| 可以中断、续读 | 读过的不再读；有花费上限 | 不重复花钱 |

### 公司里什么情况怎么做
- **用外部数据前，先登记来源、版本、许可**，写清楚能用来做什么、不能做什么
- 要调用付费接口的数据处理，**先试读 10 份看单价**，再读全部；做成可以断点续跑

### 执行
已经读完。下面三条命令都不花钱：重新确认 PMID 对照表、确认没有新的要读、看这批病人的知识库覆盖：

In [ ]:
run("uv run raredx data pmc-ids")
run("uv run raredx data read-rarearena")
run("uv run raredx eval coverage --split rarearena-dev --show 3")

### 结果怎么看
- 第一条：9,581 篇论文都有 PMID，这次查了 0 篇（之前都查过了）
- 第二条：498 道题（2 份报告读不出任何症状，放在一边）；这次读了 0 份，花费 $0
- 第三条：这 498 道题，**只读 OMIM 的知识库，184 道（37%）"有可能答对"；合并 Orphanet 以后 498 道全部有可能答对**（抽样时就要求诊断在合并后的知识库里）。剩下 314 道（63%）的诊断，在 OMIM 里找不到完全相同的病：有的是感染、肿瘤这类 OMIM 不收的病（上面列出的腹膜假黏液瘤、军团病），有的是 OMIM 只收了它的几个亚型（上面列出的厚皮性骨膜病）
- **一段插曲**：这条命令第一次跑出来，"只读 OMIM"那一列是 **0**。可第 3 步里只读 OMIM 的工具，前 10 名答对了 7.6%，两个数对不上。查下来是这条命令的错：病例报告的诊断写的是 Orphanet 编号，判分时会按 Mondo 换算成"同一个病"的 OMIM 编号，这条命令却没换算。改好了，也加了测试；其他四类病人的数没有变。**公司里的做法：两个数对不上时，先查量法**

看一道真题：

In [ ]:
import json
cases = [json.loads(l) for l in open(ROOT / "data/processed/rarearena-rds/e348a29/cases.jsonl")]
case = next(c for c in cases if c["id"] == "RareArena_10023067-1")
print(case["id"], case["pmid"], "确诊：", case["disease_label"], case["disease_id"])
print(len(case["hpo_ids"]), "个症状：", "; ".join(case["hpo_labels"]))

- 这是一位军团病（一种肺部细菌感染）病人：腹泻、低血压、心动过速、低钠血症……25 个症状
- 军团病在 Orphanet 里算罕见病，在 OMIM 里没有。只读 OMIM 的工具，根本不可能答出它
- 症状里也有和这次病无关的（双相情感障碍、高血压是病人原有的病）。读病历只管"病人有没有"，不管"和这次的病有没有关"；真实门诊的病人也是这样

### 面试怎么说
> "Our development data was almost all published single-gene cases. I added RareArena's case reports — registered with version, checksum and licence, PMC ids mapped to PMIDs so each report's own paper is masked, 500 drawn by fixed rules and read into HPO terms for about a dollar. A third are relatively common rare diseases, which our held-out set barely has; it is development data only, since the model may have read the papers."

---

## 第 3 步 · 比较排序方法

### 在干什么
在五类开发病人上，比较不用模型的"先排名再逐条核对"的各种做法，决定工具的默认排序和 Orphanet 开关。

### 背景
三样东西各有几种选法，一共 18 种组合：

| 选什么 | 选项 |
|---|---|
| 相似度 | F：两个方向平均（现在的）；O：单向；FO：两者用倒数排名融合 |
| 前 30 名怎么重排 | N：不重排；C：按解释了几个症状（现在的）；I：按解释的症状的信息量 |
| 知识库 | OMIM；合并 Orphanet |

**规矩是事先定好的**（写在设计文档里）：和现在的默认做法逐题比，**哪类病人都不能显著变差（第 1 名或前 10 名），至少两类病人显著变好**，才算通过。

### 用了什么技术，为什么
- `raredx eval rankers`（[`evals/offline.py`](../../src/raredx/evals/offline.py)）：全部在本机算，每道题照常遮挡，判分认 Mondo 的"同一个病"
- 逐题配对、符号检验（第 28 章讲过）

### 公司里什么情况怎么做
- **一次比很多种做法时，规矩要事先定，而且不能看了结果再改**。不然 18 种里总有一种碰巧好看
- 结论要看"在哪类数据上好"，不只看总分

### 执行
约 4 分钟：

In [ ]:
run("uv run raredx eval rankers")

### 结果怎么看
第一张表是每种做法的第 1 名 / 前 10 名命中率，第二张是和默认做法逐题比（`*` 是显著），最后一列 `PASS` 是通过了规矩。

- **只有一种通过：合并 Orphanet + 单向相似度 + 不重排（merged O N）**。第 1 名：RAMEDIS 49 : 25、病例报告 26 : 5；前 10 名：病例报告 95 : 8。都显著变好，没有哪类显著变差
- 它也有变差的地方，只是不显著：dev-200 前 10 名少了 3.5 个点（6 : 13），MME 前 10 名少了 2.5 个点（2 : 3）。记下来，付费运行时盯着看
- **要老实说的一点**：我先在原来四类病人上比过，**一种都没通过**。病例报告这一类是那之后才加的：设计文档原本把这个病例库放在"以后再说"，你问"数据够不够"以后，为了补"缺相对常见的病"这个空白才加进来。规矩没有改，加它的理由也和排序结果无关；**但它确实是看过一次结果之后才加的，所以这次"通过"要打个折扣**，最后靠新的外部考卷确认
- 为什么加了这一类就通过：病例报告里 63% 的题，诊断在 OMIM 里找不到完全相同的病（第 2 步）。不合并，工具前 10 名只有 7.6% 有正确答案；合并后 25.1%。所以**合不合并 Orphanet，很大程度取决于系统面对什么样的病人**。我们的新考卷正是病例报告
- 表里还能看出三种做法各偏向什么：
  - 两个方向平均（F）偏向**症状表短**的病（第 1 步讲过）
  - "按解释了几个症状重排"（C）偏向**症状表长**的病：记得越全，能解释的症状越多
  - 单向（O）也偏向**症状表长**的病：病记的症状越多，越容易给病人的每个症状找到像的
  - 所以 F 配 C，两种偏向抵消一部分，这是旧的默认做法 F C 好用的一个原因；O 再配 C 就偏过头了，O 不重排（O N）反而最好

**决定**：新流程里 **Orphanet 开关打开**；排序工具多给一路单向排序的候选（第 4 步）。"先排名再逐条核对"这个对照系统保持原样，方便和以前比。

### 面试怎么说
> "I compared eighteen no-model ranking configurations against a rule fixed beforehand: no source significantly worse, at least two significantly better. On our four original sources none passed. I then added a fifth source, published case reports, to fill a known gap — relatively common rare diseases — and there merged Orphanet knowledge with one-directional ranking passed, because 63% of those diagnoses have no exact OMIM equivalent. The source was added after I had seen the first result, so I treat the pass as provisional until the new external exam."

---

## 第 4 步 · 比较候选路线

### 在干什么
量一量：**agent 能看到的候选里，有正确答案的比例**，对不同的"提名路线"组合各是多少。

### 背景
新流程（下一章）里，候选来自好几路，每路 10 个：

| 路线 | 是什么 |
|---|---|
| M | 模型自己先猜（从记下的"直接猜"回答里读，不重新问） |
| F | 按症状排序：两个方向平均 |
| O | 按症状排序：单向 |
| C | **新的：相似的已发表病例报告**。拿病人的症状去 1 万份病例报告里检索，取最像的那些报告的诊断；病人自己那篇论文永远不返回。DeepRare 的消融实验里，这一类是贡献最大的单项；但在 HMS 上完全没帮助，在已发表病例上的提升可能有一部分是近似重复的病例漏了进来 |
| P | 相似的 Phenopacket 病人（第 17 章） |

### 用了什么技术，为什么
- C 路线用的是第 18 章同一套检索：稠密向量（BGE）加关键词（BM25）混合检索。1 万份报告的向量在本机算好存起来（`raredx data case-index`，已经做完，不花钱）
- `raredx eval routes`：每一路都在本机算；M 从记录里读。病例报告那一类没有"直接猜"的记录（还没跑过），M 那几列是 `-`

### 公司里什么情况怎么做
- 加一个检索来源之前，**先量它对召回的贡献**：它找到的，别的来源是不是已经找到了
- 每加一路都有代价（候选更多、更难选、更费 token），所以**只加真正补了空白的**

### 执行
约 8 分钟，跑两次：先用合并 Orphanet 的知识库（第 3 步的决定），再用只读 OMIM 的知识库对照（v1.1 用的就是它）：

In [ ]:
run("RAREDX_ORPHANET=1 uv run raredx eval routes")
run("uv run raredx eval routes")

### 结果怎么看
上面一张是合并 Orphanet 的，下面一张是只读 OMIM 的。先看上面一张：

- **单向排序 O 这一路，五类病人上都有补充**：F+O 比 F 多 5–31 个点（RAMEDIS 最多）
- **病例报告 C 这一路，补的正是"病例报告式"的病人**：病例报告那一类，F+C 比 F 高 18 个点；HMS 在 M+F 上再加 C，多 5.7 个点；dev-200、RAMEDIS、MME 基本不变
- **相似病人 P 这一路，只在 dev-200 上有用**：病例库就是 Phenopacket 的病人，和 dev-200 是同一类
- **不用模型的四路（F+O+C+P）**：病例报告那一类 43.4%，只用 F 是 16.3%
- **五路都加上（M+F+O+C+P）**：dev-200 66.5%、HMS 68.2%、RAMEDIS 85.7%、MME 90.0%

再和下面一张比：

- **v1.1 看到的候选，大致是只读 OMIM 的 M+F**：57.0%、59.1%、83.3%、85.0%。新的五路比它多 2–10 个点
- **合并 Orphanet 在原来四类上有一点代价**：同样五路，合并后比只读 OMIM 低 0.4–2.5 个点（比如 MME 92.5% → 90.0%）。换来的是病例报告那一类：不用模型的四路从 29.9% 到 43.4%
- 只读 OMIM 时那个 29.9% 还偏乐观：病例库 1 万份报告的诊断，只有 18% 在只读 OMIM 的知识库里，C 路线提名的病大多查不到症状，只能靠模型自己判断。合并后是 55%：病例报告这一类的正确答案都查得到，但 C 提名的其他病仍有 45% 查不到。新流程要定怎么处理它们（第 34 章）

**决定**：新流程从 M、F、O、C、P 五路提名，打开 Orphanet。新外部卷是病例报告，这个交换值得；原来四类上的小代价，付费运行时盯着看。

**工具够不够**：和 v1.1 看到的候选比，多加三路让候选全 2–10 个点；病例报告那一类，不用模型的部分从只读 OMIM 的 F（5.4%）到合并后的四路（43.4%）。这些几乎不花钱。剩下找不到的，大头是"遮挡后根本答不了"（dev-200 错题的 35%），那是考法决定的，加工具也解决不了。

### 面试怎么说
> "I treated nomination as recall: for each mix of routes I measured, offline, how often the confirmed disease is among the candidates. The case-report route — retrieval over 10,000 published reports, the patient's own paper excluded — adds little on single-gene cases but 18 points on case-report patients, the kind our next exam uses. On the four older sources all five routes hold the answer for 66–90% of cases, against 57–85% for what v1.1 saw; merging Orphanet costs up to 2.5 points there, which I accept for the case-report patients and watch in the next run."

---

## 第 5 步 · 结论：新流程的输入

| 决定 | 依据 |
|---|---|
| Orphanet 开关在新流程里**打开** | 第 3 步唯一通过规矩的做法是合并的；病例报告式的病人离不开它。代价：原来四类的候选少 0.4–2.5 个点 |
| 候选从 **M、F、O、C、P 五路**提名 | 第 4 步：每一路都至少在一类病人上补了别人没找到的 |
| "先排名再逐条核对"保持原样 | 它是对照系统，改了就和以前比不了 |
| 病例报告这一类缺"模型自己先猜"的记录 | 在 ci-20 试跑以后，和其他花钱的步骤一起报价 |

**设计文档里这一步（S1）还有三样没做**，都不花钱：
- **R2 似然比打分**（LIRICAL / PPO 那一类）：按症状在这个病里出现的频率打分，病人明确没有、而这个病常有的症状扣分。文献里它对症状少的病人最好；可另一篇同样遮挡了病例自己论文的预印本里，它反而不如 Resnik。所以要在我们的遮挡条件下比
- **K1 亚型折叠**：同一个病的好几个亚型挤占候选位置（dev-200 的前 10 名里，20% 的位置是重复的亚型）
- **按症状数分层报告**；RAMEDIS 里那 148 道乱码的苯丙酮尿症单列

再往后：搭固定流程（模型先答 → 五路提名 → 证据表 → 一次终审 → 患病率守门 → 两两裁决 → 输出），设好每次调用的输出上限，在 ci-20 上试跑。

---

## 本章小结

| 步骤 | 用的技术 | 在公司里的作用 |
|---|---|---|
| 1 单向相似度 | 祖先闭包索引；和 hpo3 逐个对数的测试 | 自己实现的算法，先对上可信的实现 |
| 2 第五类开发病人 | 登记数据；PMC → PMID；规则抽样；读病历；断点续读 | 用外部数据补开发数据的空白 |
| 3 比较排序方法 | 离线、遮挡；18 种组合；事先定的规矩 | 一次比很多种，不看结果改规矩 |
| 4 比较候选路线 | 每路召回、组合召回 | 先量召回，只加补空白的来源 |
| 5 结论 | — | 新流程的输入都由数据定 |

**一句话**：不花模型的钱，补上了"不像 Phenopacket"的开发病人，量出了 Orphanet 开关该开、候选该从五路提名；在原来四类病人上，正确答案出现在 agent 候选里的比例从 57–85% 提到 66–90%；这次"通过"是加了第五类病人以后才有的，要靠新外部卷确认。

## 面试题

> **Q: Your agent seemed limited by its tools. How did you decide which retrieval routes to add without spending money?**
>
> A: "I separated recall from ranking. For each route — the model's own answer read from recorded runs, two phenotype rankings, similar published case reports, similar curated patients — I measured offline how often the confirmed disease is among its candidates, per data source. A route stays only if it adds something the others miss; the case-report route adds 18 points on case-report patients, about 6 on clinic patients, and almost nothing on single-gene cases."

> **Q: Isn't adding a data source after seeing results a way of moving the goalposts?**
>
> A: "Partly, and I say so. The rule did not change, and the source was added for a reason unrelated to the ranking result: our development data lacked relatively common rare diseases. But it was added after I had seen that nothing passed on four sources, so I treat the pass as provisional — the decision is tested again on a new external exam the system has never seen."

## 下一章

**33 · RAG 重做**（你问"RAG 的资料是不是不全"之后加的一章）：更多资料、完整流程、分层评测。

**34 · 似然比打分和亚型折叠**（不花钱）：把设计里这一步剩下的 R2、K1 补上，用同样的规矩、同样的五类病人离线比。

**35 · 固定流程和 ci-20 试跑**：用 LangGraph 搭出新的固定流程，设好输出上限和缓存友好的提示词，在 ci-20 上试跑：查错、量单价，然后报价。